<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/5_1_cosine_similarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 코사인 유사도 튜토리얼 자료입니다.  

링크 : https://wikidocs.net/24603

# 1. 코사인 유사도(Cosine Similarity)

In [24]:
from numpy import dot
from numpy.linalg import norm
import numpy as np

In [25]:
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [26]:
doc1 = np.array([0,1,1,1])
doc2 = np.array([1,0,1,1])
doc3 = np.array([2,0,2,2])

In [27]:
print('문서 1과 문서2의 유사도 :',cos_sim(doc1, doc2))
print('문서 1과 문서3의 유사도 :',cos_sim(doc1, doc3))
print('문서 2와 문서3의 유사도 :',cos_sim(doc2, doc3))

문서 1과 문서2의 유사도 : 0.6666666666666667
문서 1과 문서3의 유사도 : 0.6666666666666667
문서 2와 문서3의 유사도 : 1.0000000000000002


# 2. 유사도를 이용한 추천 시스템 구현하기

In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
data = pd.read_csv('/content/movies_metadata.csv', low_memory=False)
data.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [40]:
# 상위 2만개의 샘플을 data에 저장
data = data.head(20000)

In [42]:
# overview 열에 존재하는 모든 결측값을 전부 카운트하여 출력
print('overview 열의 결측값의 수:',data['overview'].isnull().sum())

overview 열의 결측값의 수: 0


In [45]:
# 상위 5개 데이터를 직접 눈으로 확인
print(data['overview'].head())

# 전체 행의 개수 확인
print('전체 데이터 수:', len(data))

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: object
전체 데이터 수: 20000


In [44]:
# 결측값을 빈 값으로 대체
data['overview'] = data['overview'].fillna('')

In [46]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬의 크기(shape) :',tfidf_matrix.shape)

TF-IDF 행렬의 크기(shape) : (20000, 47487)


20,000개의 영화를 47,487개의 단어로 분석.
- 20,000(행, Row) 내가 가진 영화 줄거리의 개수
- 47,487(열, Column) 서로 다른 단어의 총 개수

In [48]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

- 첫 번째 tfidf_matrix: 기준이 되는 영화들 (20,000개)
- 두 번째 tfidf_matrix: 비교 대상이 되는 영화들 (20,000개)
- 결과: 20,000 x 20,000 크기의 거대한 점수판 생성

In [50]:
print('코사인 유사도 연산 결과 :',cosine_sim.shape)

코사인 유사도 연산 결과 : (20000, 20000)


In [51]:
title_to_index = dict(zip(data['title'], data.index))

In [52]:
# 영화 제목 Father of the Bride Part II의 인덱스를 리턴
idx = title_to_index['Father of the Bride Part II']
print(idx)

4


In [54]:
# 선택한 영화 제목 입력시 코사인 유사도를 통해 가장 overview가 유사한 10개 영화 생성
def get_recommendations(title, cosine_sim=cosine_sim):
    # 선택한 영화 타이블로부터 해당 영화의 인덱스 받아오기
    idx = title_to_index[title]

    # 해당 영화와 모든 영화와의 유사도 가져오기
    sim_scores = list(enumerate(cosine_sim[idx]))

    # 유사도에 따라 영화들을 정렬
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 가장 유사한 10개의 영화 받아오기
    sim_scores = sim_scores[1:11]


    # 가장 유사한 10개 영화의 인덱스를 얻는다
    movie_indices = [idx[0] for idx in sim_scores]

    # 가장 유사한 10개 영화의 제목을 리턴
    return data['title'].iloc[movie_indices]

In [55]:

# 다크 나이트 라이즈와 overview가 유사한 영화 찾기
get_recommendations('The Dark Knight Rises')

,title
12481,The Dark Knight
150,Batman Forever
1328,Batman Returns
15511,Batman: Under the Red Hood
585,Batman
9230,Batman Beyond: Return of the Joker
18035,Batman: Year One
19792,"Batman: The Dark Knight Returns, Part 1"
3095,Batman: Mask of the Phantasm
10122,Batman Begins
